# Voorbereiding fictief Belgisch register

Dit notebook genereert alle data (rijksregister, bisregister, families) op basis van `config.json`  
en slaat ze op als JSON-bestanden in `Generated Data/`.

**Stappen:**
1. Instellingen (prefix, paden)
2. Config tonen – wat staat er gepland?
3. Alle data opnieuw aanmaken
4. Opslaan
5. Rapport – wat staat er nu in de bestanden?

In [ ]:
import sys
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

ROOT      = Path().resolve().parent if Path().resolve().name == 'Notebooks' else Path().resolve()
SCRIPTS   = ROOT / 'Scripts'
SAVE_DIR  = ROOT / 'Generated Data'
CFG_PATH  = ROOT / 'config.json'

sys.path.insert(0, str(SCRIPTS))
SAVE_DIR.mkdir(exist_ok=True)

# ── prefix voor de output-bestanden ──────────────────────────────────────────
PREFIX = 'metaworld'

print(f'ROOT     : {ROOT}')
print(f'SAVE_DIR : {SAVE_DIR}')
print(f'PREFIX   : {PREFIX}')

## 1. Config — wat staat er gepland?

In [2]:
with open(CFG_PATH, encoding='utf-8') as f:
    cfg = json.load(f)

counts = cfg['generation']['counts']
meta   = cfg['meta']

print('=== CONFIG OVERZICHT ===')
print(f"  Seed              : {meta['seed']}")
print(f"  Referentiedatum   : {meta['creation_date']}")
print()
print('  Generatieparameters:')
print(f"    Rijksregister   : {counts['rijksregister']:,} personen")
print(f"    Bisregister     : {counts['bisregister']:,} personen")
print(f"    Stamfamilies    : {counts['stamfamilies']} koppels")
print(f"    Stamsingles     : {counts['stamsingles']} personen")
print()
print('  Demografische parameters:')
print(f"    Aandeel man     : {cfg['demographics']['gender_ratio']['male']:.1%}")
print(f"    Overlijdensfr.  : {cfg['generation']['deceased_fraction']:.3%}")

=== CONFIG OVERZICHT ===
  Seed              : 42
  Referentiedatum   : 2026-05-01

  Generatieparameters:
    Rijksregister   : 100,000 personen
    Bisregister     : 10,000 personen
    Stamfamilies    : 300 koppels
    Stamsingles     : 60 personen

  Demografische parameters:
    Aandeel man     : 49.4%
    Overlijdensfr.  : 0.800%


## 2. Data aanmaken

De volgende cellen genereren alle data opnieuw op basis van de config.  
**Pas op:** dit overschrijft de bestaande bestanden na het opslaan (stap 3).

In [3]:
from generate_register import generate_rijksregister, generate_bisregister

print(f"Rijksregister genereren ({counts['rijksregister']:,} personen)...")
t0 = datetime.now()
rr = generate_rijksregister()
print(f"  ✓ {len(rr):,} rijen — {(datetime.now()-t0).total_seconds():.1f}s")
rr.head(3)

Rijksregister genereren (100,000 personen)...
  ✓ 100,000 rijen — 1.7s


,rijksregisternummer,voornaam,familienaam,geslacht,geboortedatum,geboorteplaats,geboorteland,nationaliteit,burgerlijke_staat,overlijdensdatum,adres_straat,adres_nr,adres_bus,adres_postcode,adres_gemeente,adres_provincie,adres_gewest
0,24.04.08-998.02,Birgit,Schmidt,F,2024-04-08,Antwerpen,België,Belgisch,ongehuwd,NaN,Stationsteenweg,239,NaN,2000,Antwerpen,Antwerpen,Vlaanderen
1,07.07.08-997.55,Bart,Gilles,M,2007-07-08,Brussel,België,Belgisch,ongehuwd,NaN,Kasteellaan,95,NaN,8900,Ieper,West-Vlaanderen,Vlaanderen
2,99.01.24-997.78,Rafael,Winkler,M,1999-01-24,Antwerpen,België,Belgisch,wettelijk_samenwonend,NaN,Hofplace,72,1,8000,Brugge,West-Vlaanderen,Vlaanderen


In [4]:
print(f"Bisregister genereren ({counts['bisregister']:,} personen)...")
t0 = datetime.now()
bis = generate_bisregister()
print(f"  ✓ {len(bis):,} rijen — {(datetime.now()-t0).total_seconds():.1f}s")
bis.head(3)

Bisregister genereren (10,000 personen)...
  ✓ 10,000 rijen — 0.2s


,bisnummer,voornaam,familienaam,geslacht,geslacht_gekend,geboortedatum,geboorteplaats,geboorteland,nationaliteit,burgerlijke_staat,adres_straat,adres_nr,adres_bus,adres_postcode,adres_gemeente,adres_provincie,adres_gewest
0,26.44.24-997.39,Younes,Senhaji,M,True,2026-04-24,Montréal,Canada,Marokkaans,ongehuwd,Stationrue,619,NaN,6000,Charleroi,Henegouwen,Wallonië
1,65.31.19-997.15,Li,Huynh,X,False,1965-11-19,Aleppo,Syrië,Aziatisch,gehuwd,Parkchaussée,251,NaN,3000,Leuven,Vlaams-Brabant,Vlaanderen
2,97.28.25-997.01,Emile,Gilles,X,False,1997-08-25,Luik,België,Latijns_Amerikaans,ongehuwd,Molenchaussée,355,NaN,1030,Schaarbeek,Brussel,Brussel


In [ ]:
from generate_families import generate_families

print(f"Families genereren ({counts['stamfamilies']} stamkoppels + {counts['stamsingles']} stamsingles)...")
t0 = datetime.now()
families = generate_families(rr_df=rr, bis_df=bis)
print(f"  ✓ {len(families):,} personen in familiessysteem — {(datetime.now()-t0).total_seconds():.1f}s")
print(f"  ✓ rr en bis bijgewerkt (burgerlijke_staat, familienaam, adres)")

## 3. Opslaan naar JSON

In [6]:
def sla_df_op(df: pd.DataFrame, soort: str) -> Path:
    pad = SAVE_DIR / f'{PREFIX}_{soort}.json'
    export = df.drop(columns=['leeftijd', 'n_kinderen'], errors='ignore').copy()
    for col in export.select_dtypes(include='datetime64').columns:
        export[col] = export[col].dt.strftime('%Y-%m-%d')
    export.to_json(pad, orient='records', force_ascii=False, indent=2)
    return pad

pad_rr  = sla_df_op(rr,  'rijksregister')
pad_bis = sla_df_op(bis, 'bisregister')

pad_fam = SAVE_DIR / f'{PREFIX}_families.json'
with open(pad_fam, 'w', encoding='utf-8') as f:
    json.dump(families, f, ensure_ascii=False, indent=2)

total_kb = sum(p.stat().st_size for p in [pad_rr, pad_bis, pad_fam]) / 1024
print(f'Opgeslagen:')
print(f'  {pad_rr.name}  ({pad_rr.stat().st_size/1024:.0f} KB)')
print(f'  {pad_bis.name}  ({pad_bis.stat().st_size/1024:.0f} KB)')
print(f'  {pad_fam.name}  ({pad_fam.stat().st_size/1024:.0f} KB)')
print(f'  Totaal: {total_kb:.0f} KB')

Opgeslagen:
  register_rijksregister.json  (51257 KB)
  register_bisregister.json  (5023 KB)
  register_families.json  (1966 KB)
  Totaal: 58245 KB


## 5. Loopbanen genereren

Genereert Dimona-, RSVZ- en RVW-bestanden voor alle personen.
**Opgelet:** dit duurt enkele minuten bij grote registers.

In [ ]:
from generate_careers import generate_careers

print("Loopbanen genereren...")
t0 = datetime.now()
dimona, rsvz, rvw = generate_careers(rr_df=rr, bis_df=bis)
elapsed = (datetime.now()-t0).total_seconds()
print(f"  ✓ Dimona  : {len(dimona):,} aangiften")
print(f"  ✓ RSVZ    : {len(rsvz):,} aansluitingen")
print(f"  ✓ RVW     : {len(rvw):,} perioden")
print(f"  ✓ Personen: {len({r['rijksregisternummer'] for r in dimona+rsvz+rvw}):,} uniek  ({elapsed:.1f}s)")

In [ ]:
# Opslaan
for data, soort in [(dimona, "dimona"), (rsvz, "rsvz"), (rvw, "rvw")]:
    pad = SAVE_DIR / f"{PREFIX}_{soort}.json"
    with open(pad, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  → {pad.name}  ({pad.stat().st_size/1024:.0f} KB)")

## 4. Rapport — wat staat er in de bestanden?

Deze sectie leest de opgeslagen JSON-bestanden en toont een overzicht.  
Je kunt deze cellen ook los uitvoeren zonder data opnieuw te genereren.

In [ ]:
def rapporteer_bestanden(save_dir: Path, prefix: str) -> None:
    """
    Leest de JSON-bestanden voor het gegeven prefix en print een volledig rapport.
    Kan ook worden aangeroepen zonder data te genereren.
    """
    rr_pad  = save_dir / f'{prefix}_rijksregister.json'
    bis_pad = save_dir / f'{prefix}_bisregister.json'
    fam_pad = save_dir / f'{prefix}_families.json'

    SEP = '─' * 52

    with open(CFG_PATH, encoding='utf-8') as f:
        cfg_r = json.load(f)
    ref = pd.Timestamp(cfg_r['meta']['creation_date'])

    def _leeftijd(df, datumkol='geboortedatum'):
        df = df.copy()
        df[datumkol] = pd.to_datetime(df[datumkol])
        df['leeftijd'] = ((ref - df[datumkol]).dt.days // 365).clip(0, 120)
        return df

    # ── Rijksregister ──────────────────────────────────────────────────────────
    print(SEP)
    print('  RIJKSREGISTER')
    print(SEP)
    if not rr_pad.exists():
        print('  ✗ Bestand niet gevonden:', rr_pad)
    else:
        ts = datetime.fromtimestamp(rr_pad.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        rr_df = _leeftijd(pd.read_json(rr_pad, convert_dates=False))
        levend = rr_df['overlijdensdatum'].isna()
        print(f'  Bestand          : {rr_pad.name}  (opgeslagen {ts})')
        print(f'  Grootte          : {rr_pad.stat().st_size/1024:.0f} KB')
        print(f'  Totaal personen  : {len(rr_df):,}')
        print(f'  Levend           : {levend.sum():,}  ({levend.mean():.1%})')
        print(f'  Overleden        : {(~levend).sum():,}  ({(~levend).mean():.1%})')
        print(f'  Geslacht M/F     : {(rr_df.geslacht=="M").sum():,} / {(rr_df.geslacht=="F").sum():,}')
        print(f'  Gem. leeftijd    : {rr_df.loc[levend, "leeftijd"].mean():.1f} jr (levenden)')
        print(f'  Buitenl. geboorte: {(rr_df.geboorteland != "België").mean():.1%}')
        print('  Nationaliteit top 5:')
        for nat, cnt in rr_df['nationaliteit'].value_counts().head(5).items():
            print(f'    {nat:<22}: {cnt:,}  ({cnt/len(rr_df):.1%})')

    # ── Bisregister ───────────────────────────────────────────────────────────
    print()
    print(SEP)
    print('  BISREGISTER')
    print(SEP)
    if not bis_pad.exists():
        print('  ✗ Bestand niet gevonden:', bis_pad)
    else:
        ts = datetime.fromtimestamp(bis_pad.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        bis_df = _leeftijd(pd.read_json(bis_pad, convert_dates=False))
        print(f'  Bestand          : {bis_pad.name}  (opgeslagen {ts})')
        print(f'  Grootte          : {bis_pad.stat().st_size/1024:.0f} KB')
        print(f'  Totaal personen  : {len(bis_df):,}')
        print(f'  Geslacht M/F/X   : {(bis_df.geslacht=="M").sum():,} / {(bis_df.geslacht=="F").sum():,} / {(bis_df.geslacht=="X").sum():,}')
        print(f'  Gem. leeftijd    : {bis_df["leeftijd"].mean():.1f} jr')
        print(f'  Belgisch geboren : {(bis_df.geboorteland == "België").mean():.1%}')

    # ── Families ──────────────────────────────────────────────────────
    print()
    print(SEP)
    print('  FAMILIES')
    print(SEP)
    if not fam_pad.exists():
        print('  ✗ Bestand niet gevonden:', fam_pad)
    else:
        ts = datetime.fromtimestamp(fam_pad.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        fam_df = pd.read_json(fam_pad, convert_dates=False)
        fam_df['n_kinderen'] = fam_df['kinderen_rr'].apply(
            lambda x: len(x) if isinstance(x, list) else 0
        )
        koppels  = fam_df['partner_rr'].notna().sum() // 2
        met_kind = (fam_df['n_kinderen'] > 0).sum()
        overl_fam = 0
        if rr_pad.exists():
            rr_tmp = pd.read_json(rr_pad, convert_dates=False)
            rr_overl = set(rr_tmp[rr_tmp['overlijdensdatum'].notna()]['rijksregisternummer'].astype(str))
            overl_fam = fam_df['rijksregisternummer'].astype(str).isin(rr_overl).sum()
        print(f'  Bestand          : {fam_pad.name}  (opgeslagen {ts})')
        print(f'  Grootte          : {fam_pad.stat().st_size/1024:.0f} KB')
        print(f'  Totaal personen  : {len(fam_df):,}')
        print(f'  Koppels          : {koppels:,}')
        print(f'  Met kinderen     : {met_kind:,}')
        print(f'  Overledenen      : {overl_fam:,}  ({overl_fam/len(fam_df):.1%})')
        print('  Personen per generatie:')
        gen_labels = {0: 'stamouders', 1: 'gen 1', 2: 'gen 2', 3: 'gen 3'}
        for g, cnt in fam_df['generatie'].value_counts().sort_index().items():
            print(f'    {g} ({gen_labels.get(g, '?'):12}): {cnt:,}')
    print()
    print(SEP)
# Rapport uitvoeren op de zojuist opgeslagen bestanden
rapporteer_bestanden(SAVE_DIR, PREFIX)

---
### Rapport op bestaand bestand (zonder generatie)

Voer enkel onderstaande cel uit om een rapport te krijgen van een bestaand bestand.

In [10]:
# Pas PREFIX_RAPPORT aan om een ander bestand te inspecteren
PREFIX_RAPPORT = PREFIX
rapporteer_bestanden(SAVE_DIR, PREFIX_RAPPORT)

────────────────────────────────────────────────────
  RIJKSREGISTER
────────────────────────────────────────────────────
  Bestand          : register_rijksregister.json  (opgeslagen 2026-05-02 07:52)
  Grootte          : 51257 KB
  Totaal personen  : 100,000
  Levend           : 99,376  (99.4%)
  Overleden        : 624  (0.6%)
  Geslacht M/F     : 49,242 / 50,758
  Gem. leeftijd    : 39.3 jr (levenden)
  Buitenl. geboorte: 5.0%
  Nationaliteit top 5:
    Belgisch              : 75,022  (75.0%)
    EU_overig             : 11,414  (11.4%)
    Marokkaans            : 3,747  (3.7%)
    Overig                : 2,183  (2.2%)
    Turks                 : 2,038  (2.0%)

────────────────────────────────────────────────────
  BISREGISTER
────────────────────────────────────────────────────
  Bestand          : register_bisregister.json  (opgeslagen 2026-05-02 07:52)
  Grootte          : 5023 KB
  Totaal personen  : 10,000
  Geslacht M/F/X   : 4,190 / 4,303 / 1,507
  Gem. leeftijd    : 39.4 jr
 